# Actividad 2: Modelos de Predicción y Clasificación

### Grupo: AE_Grupo_6
Integrantes:
* Juan Fernando Gaviria Higuita
* Juan David Ramírez García

## 0. Construcción Automática del DatFrame

Enlace en huggingface del dataset:[HealthRiskPredictor](https://huggingface.co/datasets/infinite-dataset-hub/HealthRiskPredictor)

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

In [2]:
loader = HFDatasetLoader(
    dataset_name="infinite-dataset-hub/HealthRiskPredictor",
    split="train",
    filename="datos/health_risk_predictor.csv",
)

In [3]:
df = loader.load()
df.head()

,idx,age,gender,weight,height,exercise_frequency,smoking_status,diet_type,family_history_of_diseases,medical_history,risk_level
0,0,29,male,85,180,4,non-smoker,balanced,no,none,low
1,1,35,female,70,165,2,smoker,vegan,yes,hypertension,high
2,2,42,male,95,175,1,non-smoker,omnivore,no,none,medium
3,3,58,female,78,160,3,non-smoker,vegetarian,yes,diabetes,high
4,4,33,male,88,178,2,non-smoker,balanced,no,none,low


## 1. Evidencia de preprocesamiento

Revisando estructura del dataFrame

In [4]:
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"Valores nulos totales: {df.isnull().sum().sum()}\n")

print("--- Información de tipos de datos y memoria ---")
df.info()

Filas: 91
Columnas: 11
Filas duplicadas: 0
Valores nulos totales: 0

--- Información de tipos de datos y memoria ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   idx                         91 non-null     int64 
 1   age                         91 non-null     int64 
 2   gender                      91 non-null     object
 3   weight                      91 non-null     int64 
 4   height                      91 non-null     int64 
 5   exercise_frequency          91 non-null     int64 
 6   smoking_status              91 non-null     object
 7   diet_type                   91 non-null     object
 8   family_history_of_diseases  91 non-null     object
 9   medical_history             91 non-null     object
 10  risk_level                  91 non-null     object
dtypes: int64(5), object(6)
memory usage: 7.9+ KB


El dataFrame no tiene datos entonces no haremos limpieza de nulos y de duplicados o imputación de datos.

In [5]:
# Eliminando la columna redundante 'idx' porque es la misma que el índice del DataFrame
df_clean= df.drop(columns=["idx"])
# Verificación de la lista de columnas
print(f"Verificando que la columna 'idx' ha sido eliminada:\n{df_clean.columns}")

Verificando que la columna 'idx' ha sido eliminada:
Index(['age', 'gender', 'weight', 'height', 'exercise_frequency',
       'smoking_status', 'diet_type', 'family_history_of_diseases',
       'medical_history', 'risk_level'],
      dtype='object')


In [6]:
# Calculando el Índice de Masa Corporal (IMC)
df_clean.loc[:, "bmi"] = df_clean["weight"] / ((df_clean["height"] / 100) ** 2)
print("Se ha calculado el Índice de Masa Corporal (IMC) y se ha agregado como una nueva columna 'bmi'.")
df_clean.head(2)

Se ha calculado el Índice de Masa Corporal (IMC) y se ha agregado como una nueva columna 'bmi'.


,age,gender,weight,height,exercise_frequency,smoking_status,diet_type,family_history_of_diseases,medical_history,risk_level,bmi
0,29,male,85,180,4,non-smoker,balanced,no,none,low,26.234568
1,35,female,70,165,2,smoker,vegan,yes,hypertension,high,25.711662


In [7]:
# Filtrando únicamente las variables numéricas
df_num = df_clean.select_dtypes(include=['number'])

# Calculando la matriz de correlación
matriz_corr = df_num.corr()

#  correlación de todas las variables frente a 'bmi' ordenadas
print("Correlación lineal con la variable objetivo (bmi):")
print(matriz_corr['bmi'].sort_values(ascending=False))

Correlación lineal con la variable objetivo (bmi):
bmi                   1.000000
weight                0.762834
age                   0.624914
height                0.122795
exercise_frequency   -0.529000
Name: bmi, dtype: float64


Calculamos las correlaciones para medir la fuerza de la relación lineal entre cada variable y el **bmi**, lo que nos permitió identificar qué características aportan información predictiva real. Descartamos explícitamente **weight** (y **height**) por representar un riesgo de fuga de datos (*data leakage*), ya que son los componentes exactos con los que se calcula matemáticamente este índice. A partir de este análisis, seleccionamos la variable con la mejor correlación válida (**age**, con 0.62) para construir un modelo de regresión lineal simple que estime el **IMC** en función de la edad.

In [8]:
print("=== COMPARACIÓN DE DATAFRAMES ===")
print(f"Dimensiones Original (df_raw): {df.shape}")
print(f"Dimensiones Limpio   (df_clean): {df_clean.shape}\n")

columnas_eliminadas = set(df.columns) - set(df_clean.columns)
columnas_agregadas = set(df_clean.columns) - set(df.columns)

print(f"Columnas eliminadas: {columnas_eliminadas}")
print(f"Columnas agregadas:   {columnas_agregadas}")

=== COMPARACIÓN DE DATAFRAMES ===
Dimensiones Original (df_raw): (91, 11)
Dimensiones Limpio   (df_clean): (91, 11)

Columnas eliminadas: {'idx'}
Columnas agregadas:   {'bmi'}


## 2. Modelo de regresión lineal e interpretación

La pregunta que queremos responder con este nuevo modelo es: **"¿Podemos estimar el Índice de Masa Corporal (IMC) de una persona conociendo únicamente su edad, y cuánto tiende a aumentar ese IMC por cada año de vida?"**

Las variables que utilizamos para estructurar esta pregunta se definen así:

* Variable Objetivo o "y": El IMC (bmi). Es el resultado que queremos calcular o predecir. Se llama "dependiente" porque, para este modelo, asumimos que su valor depende o está influenciado por la edad del paciente.

* Variable Predictora  "X": La Edad (age). Es la información o pista que le entregamos al modelo matemático para que intente hacer su predicción.

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import numpy as np

X, y = df_clean[["age"]], df_clean["bmi"] # preparar datos

#Inicializando modelo
modelo = LinearRegression() 

# Dividiendo datos 80% para entrenamiento y 20% para test.
X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size=0.2, random_state=42
)

# Entrenado el modelo 
modelo.fit(X_train, y_train)

#Prediciendo
predicciones = modelo.predict(X_test) 
w = modelo.coef_[0]
b = modelo.intercept_

print(f"MAE: {mean_absolute_error(y_test, predicciones):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, predicciones)):.2f}")
print(f"R2: {r2_score(y_test, predicciones):.2f}")
print(f"w: {w:.2f}")
print(f"b: {b:.2f}")

MAE: 1.59
RMSE: 1.99
R2: 0.34
w: 0.15
b: 20.55


* **R²** :  El 66% restante del IMC depende de factores que el modelo no está viendo actualmente, como los hábitos alimenticios o el nivel de actividad física.

* **MAE** :  Un MAE de 1.59 indica que, en promedio, las predicciones del modelo se desvían por 1.59 puntos de IMC respecto a la realidad, ya sea adivinando por encima o por debajo del valor real.

* **RMSE** : También mide el error en puntos de IMC, pero con una trampa estadística: castiga más fuerte los errores grandes. Como tu RMSE (1.99) es más alto que tu MAE (1.59), esto te advierte que el modelo tiene algunas predicciones donde se equivoca por bastante margen (casos atípicos), lo que empuja este promedio hacia arriba.

## Resumen automático del modelo en un archivo .txt

Importando la clase de reportes

In [10]:
# 1. Importar la CLASE (ReporteRegresion) desde tu archivo (Reporte_regresion)
from clases_po.Reporte_regresion import ReporteRegresion

# 2. Inicializar la clase correctamente
reportador = ReporteRegresion()

# 3. Generar y guardar el archivo .txt
reportador.guardar_reporte(
    y_test=y_test, 
    predicciones=predicciones, 
    modelo=modelo, 
    ruta="reportes/reporte_regresion.txt"
)

print(f"Reporte guardado en: {ruta}")

# 4. Imprimir en pantalla para verificar
reportador.imprimir_reporte(y_test, predicciones, modelo)

NameError: name 'ruta' is not defined